In [ ]:
import subprocess
import sys

import torch

print(torch.cuda.get_device_name(0))

from google.colab import drive

drive.mount('/content/drive')

from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
subprocess.run(
    [
        'git',
        'clone',
        f'https://{token}@github.com/vaibhavsadgir50/uni.git',
        '/content/uni',
    ],
    check=True,
)
sys.path.insert(0, '/content/uni')
subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'transformers',
        'datasets',
        'torch',
        'matplotlib',
    ],
    check=True,
)


In [ ]:
import torch

from T2 import build_gpt2_rsm_model, GPT2RSMConfig

cfg = GPT2RSMConfig()
model = build_gpt2_rsm_model(cfg, device='cuda')
B = 4
conv_states, domain_states = model.init_states(
    B, torch.device('cuda'), model.wte.weight.dtype
)
print(torch.cuda.memory_summary())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('total trainable parameters:', trainable)
del model, conv_states, domain_states
torch.cuda.empty_cache()


In [ ]:
%matplotlib inline
import os

from IPython.display import clear_output

# All checkpoints (t2_v1_best.pt + t2_v1_epoch_*_val*.pt) go here — persists on Google Drive.
_DRIVE_CKPT = "/content/drive/MyDrive/dpnn_g/checkpoints/t2_v1_best.pt"
if not os.path.isdir("/content/drive/MyDrive"):
    raise RuntimeError("Mount Google Drive in cell 0 before training (drive.mount).")
os.makedirs(os.path.dirname(_DRIVE_CKPT), exist_ok=True)
os.environ["T2_CHECKPOINT"] = _DRIVE_CKPT
print("Checkpoints will save under:", os.path.dirname(_DRIVE_CKPT))

from T2.train import main

main()
